# Benchmark — all models, all conditions, one run

Runs RF / GCN / GAT × {no-GDELT, concat keep-all, concat top-k} and logs every result to `results.csv`.
Run the cells top to bottom, then the **driver** cell does the whole grid.

In [53]:
import os, csv
from datetime import datetime
import numpy as np, pandas as pd, torch
import torch.nn as nn
import dgl
from dgl.nn import GATConv, GraphConv
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import warnings; warnings.filterwarnings('ignore')

GDELT_COLS = ['events_total','score_mean','score_max','score_vol',
              'goldstein_wmean','tone_mean','active_months']
BASE_COLS  = ['refYear','cmdCode','dist','gdpcap_d','gdpcap_o','pop_o','pop_d']  # no primaryValue (leak-free)
RESULTS_FILE = 'results.csv'

## Helpers: logging, GDELT merge, aggregation, metrics

In [54]:
def log_result(model, fusion, scoring, gdelt, risk_level, agg_mode, tranche, r2, mae, smape, n, notes=''):
    row = {'timestamp': datetime.now().isoformat(timespec='seconds'),
           'model': model, 'fusion': fusion, 'scoring': scoring, 'gdelt': gdelt,
           'risk_level': risk_level, 'agg_mode': agg_mode, 'tranche': tranche,
           'r2': round(float(r2),4), 'mae': round(float(mae),2),
           'smape': round(float(smape),2), 'n': int(n), 'notes': notes}
    new = not os.path.exists(RESULTS_FILE)
    with open(RESULTS_FILE,'a',newline='') as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if new: w.writeheader()
        w.writerow(row)
    return row

def _smape(yt, yp):
    d = (np.abs(yt)+np.abs(yp))/2; m = d>0
    return np.mean(np.abs(yt[m]-yp[m])/d[m])*100 if m.sum() else np.nan

def add_gdelt(agg, gdelt_file):
    '''String-key merge so reporterCode dtype is preserved.'''
    cy = pd.read_parquet(gdelt_file)
    out = agg.copy()
    out['_r'] = out['reporterCode'].astype('Int64').astype(str)
    out['_y'] = out['refYear'].astype('Int64').astype(str)
    cy['_r'] = cy['reporterCode'].astype('Int64').astype(str)
    cy['_y'] = cy['year'].astype('Int64').astype(str)
    out = out.merge(cy[['_r','_y']+GDELT_COLS], on=['_r','_y'], how='left')
    out[GDELT_COLS] = out[GDELT_COLS].fillna(0)
    return out.drop(columns=['_r','_y'])

def build_agg(df, use_gdelt, gdelt_file, agg_mode="sum"):
    grav = "first" if agg_mode == "first" else "sum"   # "first"=fix, "sum"=supervisor original
    agg = (df.groupby(['refYear','reporterCode','cmdCode'])
             .agg(primaryValue=('primaryValue','mean'), dist=('dist','first'),
                  gdpcap_d=('gdpcap_d',grav), gdpcap_o=('gdpcap_o',grav),
                  pop_o=('pop_o',grav), pop_d=('pop_d',grav)).reset_index())
    if use_gdelt: agg = add_gdelt(agg, gdelt_file)
    return agg


def report_and_log(model, use_gdelt, scoring, agg_mode, y_true, y_pred, fusion_override=None):
    fus = fusion_override if fusion_override is not None else ('concat' if use_gdelt else 'none')
    sc  = scoring if use_gdelt else 'none'
    print(f'\n=== {model} gdelt={use_gdelt} scoring={sc} agg={agg_mode} fusion={fus} — Test 2023 ===')
    for label, thr in [('all',None),('1M',1e6),('10M',1e7),('100M',1e8)]:
        m = np.ones_like(y_true,bool) if thr is None else (y_true>=thr)
        if m.sum() < 10: continue
        r2=r2_score(y_true[m],y_pred[m]); mae=mean_absolute_error(y_true[m],y_pred[m])
        sm=_smape(y_true[m],y_pred[m]); n=int(m.sum())
        print(f'  {label:4s} n={n:>6,} R2={r2:.4f} SMAPE={sm:.2f}%')
        log_result(model, fus, sc, use_gdelt, 'node', agg_mode, label, r2, mae, sm, n)

## Load + clean + split — run ONCE (shared by every model)

In [55]:
def load_and_split(path='all_products_ready.parquet'):
    # column-subset load: only what the models use (keeps 26M rows in RAM)
    cols = ['refYear','reporterCode','partnerCode','cmdCode',
            'gdpcap_o','pop_o','gdpcap_d','pop_d','dist','primaryValue']
    df = pd.read_parquet(path, columns=cols)
    for c in ['gdpcap_o','gdpcap_d','dist','pop_o','pop_d','primaryValue','refYear','cmdCode','reporterCode','partnerCode']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df['gdpcap_o']/=1e6; df['gdpcap_d']/=1e6; df['dist']/=1e3
    df = df[df['cmdCode'].notna()].copy(); df['cmdCode']=df['cmdCode'].astype(int)
    d = df.drop_duplicates(['refYear','reporterCode','partnerCode','cmdCode']).reset_index(drop=True)
    tr_full = d[d['refYear'].isin([2017,2018,2019,2020,2021,2022])].copy()
    te      = d[d['refYear']==2023].copy()
    tr_full, te = handle_missing(tr_full, te, strategy=MISSING)
    train_data, val_data = [], []
    for _, grp in tr_full.groupby('reporterCode'):
        if len(grp) < 5:
            train_data.append(grp); continue
        a,b = train_test_split(grp, test_size=0.2, random_state=42)
        train_data.append(a); val_data.append(b)
    train_data = pd.concat(train_data); val_data = pd.concat(val_data)
    print(f'train {len(train_data):,} | val {len(val_data):,} | test2023 {len(te):,}')
    return train_data, val_data, te

train_data, val_data, data_2023 = load_and_split('electronics.parquet')

train 184,025 | val 46,070 | test2023 34,721


## Models

In [56]:
class GATRegressionModel(nn.Module):
    def __init__(self, in_feats, hidden=32, heads=4):
        super().__init__()
        self.c1 = GATConv(in_feats, hidden, heads)
        self.c2 = GATConv(hidden*heads, 1, heads)
    def forward(self, g, x):
        x = torch.relu(self.c1(g, x).flatten(1))
        return self.c2(g, x).mean(1)

class GCNRegressionModel(nn.Module):
    def __init__(self, in_feats, hidden=32):
        super().__init__()
        self.c1 = GraphConv(in_feats, hidden, allow_zero_in_degree=True)
        self.c2 = GraphConv(hidden, 1, allow_zero_in_degree=True)
    def forward(self, g, x):
        x = torch.relu(self.c1(g, x))
        return self.c2(g, x).squeeze(-1)

## Graph runner (GAT or GCN) and RF runner

In [57]:
def _eval_graph(model, df_eval, cmap, sf, st, feat_cols, use_gdelt, gdelt_file,
                agg_mode="sum", fusion="concat", nt=None, bs=10000):
    d = df_eval.copy()
    for c in ['gdpcap_o','pop_o','gdpcap_d','pop_d','dist']:
        d[c] = d[c].fillna(d[c].mean())
    d['nID'] = d['reporterCode'].map(cmap); d['pID'] = d['partnerCode'].map(cmap)
    d = d.dropna(subset=['nID','pID']).copy()
    if len(d)==0: return None, None
    agg = build_agg(d, use_gdelt, gdelt_file, agg_mode)
    emap = {c:i for i,c in enumerate(agg['reporterCode'])}
    Xe = sf.transform(agg[feat_cols]); ye = st.transform(agg[['primaryValue']])
    d['eN'] = d['reporterCode'].map(emap); d['eP'] = d['partnerCode'].map(emap)
    d = d.dropna(subset=['eN','eP']); d['eN']=d['eN'].astype(int); d['eP']=d['eP'].astype(int)
    eg = dgl.graph((d['eN'].to_numpy(), d['eP'].to_numpy()))
    eg.ndata['feat'] = torch.tensor(Xe, dtype=torch.float32)
    eg = dgl.add_self_loop(eg)
    fused_model = fusion in ("blend","attention")
    model.eval(); preds=[]; N=eg.num_nodes()
    for i in range(0,N,bs):
        bn=list(range(i,min(i+bs,N))); bg=eg.subgraph(bn)
        feat = bg.ndata['feat']
        with torch.no_grad():
            if fused_model:
                out = model(bg, feat[:, :nt], feat[:, nt:])
            else:
                out = model(bg, feat)
            preds.append(out.unsqueeze(1))
    yp = st.inverse_transform(torch.cat(preds,0).view(-1,1).numpy()).flatten()
    yt = st.inverse_transform(ye).flatten()
    return yt, yp

def run_graph(kind, train_data, data_2023, use_gdelt, scoring, gdelt_file,
              fusion="concat", epochs=100, agg_mode="sum"):
    feat_cols = BASE_COLS + (GDELT_COLS if use_gdelt else [])
    agg  = build_agg(train_data, use_gdelt, gdelt_file, agg_mode)
    cmap = {c:i for i,c in enumerate(agg['reporterCode'])}
    td = train_data.copy(); td['nodeID'] = td['reporterCode'].map(cmap)
    sf, st = MinMaxScaler(), MinMaxScaler()
    Xtr = sf.fit_transform(agg[feat_cols]); ytr = st.fit_transform(agg[['primaryValue']])
    g = dgl.graph((td['nodeID'].to_numpy(), td['partnerCode'].map(cmap).to_numpy()))
    g.ndata['feat'] = torch.tensor(Xtr, dtype=torch.float32); g = dgl.add_self_loop(g)

    fused_model = fusion in ("blend","attention")
    nt = len(BASE_COLS)
    if fusion == "blend":       model = BlendGAT(nt, len(GDELT_COLS))
    elif fusion == "attention": model = AttnGAT(nt, len(GDELT_COLS))
    elif kind == "GAT":         model = GATRegressionModel(len(feat_cols))
    else:                       model = GCNRegressionModel(len(feat_cols))

    def split_feats(feat):                 # trade cols first, gdelt cols after
        return feat[:, :nt], feat[:, nt:]

    opt = torch.optim.Adam(model.parameters(), lr=0.01); crit = nn.MSELoss()
    N=g.num_nodes(); bs=10000; nb=N//bs+(N%bs>0)
    for ep in range(epochs):
        model.train()
        for i in range(nb):
            bn=list(range(i*bs,min((i+1)*bs,N))); bg=g.subgraph(bn)
            feat = bg.ndata['feat']
            logit = model(bg, *split_feats(feat)) if fused_model else model(bg, feat)
            tgt = torch.tensor(ytr[bn,0],dtype=torch.float32).view(-1,1)
            loss = crit(logit.view(-1,1),tgt)
            opt.zero_grad(); loss.backward(); opt.step()

    yt, yp = _eval_graph(model, data_2023, cmap, sf, st, feat_cols,
                         use_gdelt, gdelt_file, agg_mode, fusion=fusion, nt=nt)
    if yt is not None:
        log_fusion = fusion if use_gdelt else "none"
        report_and_log(kind, use_gdelt, scoring, agg_mode, yt, yp, fusion_override=log_fusion)
    if fusion == "blend":
        print(f"   learned alpha (geopolitics weight) = {torch.sigmoid(model.alpha).item():.3f}")
    return model

def run_rf(train_data, data_2023, use_gdelt, scoring, gdelt_file, agg_mode="sum"):
    feat_cols = BASE_COLS + (GDELT_COLS if use_gdelt else [])
    tr = build_agg(train_data, use_gdelt, gdelt_file, agg_mode).dropna(subset=feat_cols+['primaryValue'])
    te = build_agg(data_2023,  use_gdelt, gdelt_file, agg_mode).dropna(subset=feat_cols+['primaryValue'])
    sf, st = MinMaxScaler(), MinMaxScaler()
    Xtr=sf.fit_transform(tr[feat_cols]); Xte=sf.transform(te[feat_cols])
    ytr=st.fit_transform(tr[['primaryValue']]); yte=st.transform(te[['primaryValue']])
    rf = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=0).fit(Xtr, ytr.ravel())
    yp = st.inverse_transform(rf.predict(Xte).reshape(-1,1)).flatten()
    yt = st.inverse_transform(yte).flatten()
    report_and_log('RF', use_gdelt, scoring, agg_mode, yt, yp)
    return rf

## Blending technic & Attention BLENDER

In [58]:
class BlendGAT(nn.Module):
    """Weighted-blend fusion: project trade & GDELT separately, blend with a learnable alpha."""
    def __init__(self, n_trade, n_gdelt, proj=16, hidden=32, heads=4):
        super().__init__()
        self.trade_proj = nn.Linear(n_trade, proj)
        self.gdelt_proj = nn.Linear(n_gdelt, proj)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.c1 = GATConv(proj, hidden, heads)
        self.c2 = GATConv(hidden*heads, 1, heads)
    def forward(self, g, x_trade, x_gdelt):
        t = torch.relu(self.trade_proj(x_trade))
        d = torch.relu(self.gdelt_proj(x_gdelt))
        a = torch.sigmoid(self.alpha)                 # keep blend in [0,1]
        fused = a*d + (1-a)*t
        x = torch.relu(self.c1(g, fused).flatten(1))
        return self.c2(g, x).mean(1)

class AttnGAT(nn.Module):
    """Attention fusion: a small attention layer weights the two streams per node."""
    def __init__(self, n_trade, n_gdelt, proj=16, hidden=32, heads=4):
        super().__init__()
        self.trade_proj = nn.Linear(n_trade, proj)
        self.gdelt_proj = nn.Linear(n_gdelt, proj)
        self.attn = nn.MultiheadAttention(embed_dim=proj, num_heads=1, batch_first=True)
        self.c1 = GATConv(proj, hidden, heads)
        self.c2 = GATConv(hidden*heads, 1, heads)
    def forward(self, g, x_trade, x_gdelt):
        t = torch.relu(self.trade_proj(x_trade))
        d = torch.relu(self.gdelt_proj(x_gdelt))
        stack = torch.stack([t, d], dim=1)            # (N, 2, proj)
        out, _ = self.attn(stack, stack, stack)       # attend across the 2 streams
        fused = out.mean(dim=1)                        # (N, proj)
        x = torch.relu(self.c1(g, fused).flatten(1))
        return self.c2(g, x).mean(1)

## DRIVER — runs the whole grid in one go

In [59]:
KEEPALL = 'gdelt_features_by_country_year.parquet'
TOPK    = 'gdelt_features_topk.parquet'

CONDITIONS = [
    ('none',    False, None),
    ('keepall', True,  KEEPALL),
    ('topk',    True,  TOPK),
]

# --- main grid: 3 models x 3 conditions x 2 agg modes (concat fusion) ---
for agg_mode in ['sum', 'first']:
    for scoring, use_gdelt, gfile in CONDITIONS:
        run_rf   (train_data, data_2023, use_gdelt, scoring, gfile, agg_mode)
        run_graph('GCN', train_data, data_2023, use_gdelt, scoring, gfile, agg_mode=agg_mode)
        run_graph('GAT', train_data, data_2023, use_gdelt, scoring, gfile, agg_mode=agg_mode)

# --- GAT fusion sub-study: blend + attention, GDELT on, agg='first' only ---
for fusion in ['blend', 'attention']:
    for scoring, gfile in [('keepall', KEEPALL), ('topk', TOPK)]:
        run_graph('GAT', train_data, data_2023, True, scoring, gfile,
                  fusion=fusion, agg_mode='first')

print('\n=== ALL RUNS DONE ===')


=== RF gdelt=False scoring=none agg=sum fusion=none — Test 2023 ===
  all  n=   953 R2=-0.6756 SMAPE=154.51%
  1M   n=   208 R2=-0.3122 SMAPE=116.78%
  10M  n=    82 R2=-0.1980 SMAPE=79.16%
  100M n=    20 R2=-0.7003 SMAPE=56.34%

=== GCN gdelt=False scoring=none agg=sum fusion=none — Test 2023 ===
  all  n=   948 R2=0.0178 SMAPE=183.61%
  1M   n=   224 R2=0.1521 SMAPE=139.13%
  10M  n=    87 R2=0.0496 SMAPE=101.75%
  100M n=    22 R2=-1.3353 SMAPE=104.57%

=== GAT gdelt=False scoring=none agg=sum fusion=none — Test 2023 ===
  all  n=   948 R2=0.1675 SMAPE=179.93%
  1M   n=   224 R2=0.2239 SMAPE=124.96%
  10M  n=    87 R2=0.1168 SMAPE=79.64%
  100M n=    22 R2=-1.1879 SMAPE=100.18%

=== RF gdelt=True scoring=keepall agg=sum fusion=concat — Test 2023 ===
  all  n=   953 R2=-0.1557 SMAPE=152.37%
  1M   n=   208 R2=0.1137 SMAPE=116.88%
  10M  n=    82 R2=0.3314 SMAPE=80.99%
  100M n=    20 R2=0.0911 SMAPE=37.23%

=== GCN gdelt=True scoring=keepall agg=sum fusion=concat — Test 2023 ===
  

## Results table (pivot from results.csv)

In [60]:
r = pd.read_csv('results.csv')
r = r.drop_duplicates(['model','fusion','scoring','gdelt','risk_level','agg_mode','tranche'], keep='last')

# ============ MAIN GRID: model x condition (concat fusion only) ============
main = r[r.fusion.isin(['none','concat'])].copy()
for am in ['sum','first']:
    for tr in ['all','10M']:
        sub = main[(main.tranche==tr) & (main.agg_mode==am)].copy()
        sub['cond'] = np.where(~sub.gdelt, 'no-GDELT', 'GDELT/'+sub.scoring)
        print(f'\n--- agg={am} | R2 tranche {tr} ---')
        print(sub.pivot_table(index='model', columns='cond', values='r2').round(3))

# ============ FUSION SUB-STUDY: GAT only, GDELT on, agg='first' ============
gat = r[(r.model=='GAT') & (r.gdelt==True) & (r.agg_mode=='first')].copy()
for tr in ['all','1M','10M']:
    sub = gat[gat.tranche==tr]
    print(f'\n--- GAT fusion | R2 tranche {tr} ---')
    print(sub.pivot_table(index='fusion', columns='scoring', values='r2').round(3))


--- agg=sum | R2 tranche all ---
cond   GDELT/keepall  GDELT/topk  no-GDELT
model                                     
GAT            0.159       0.023     0.168
GCN           -0.196      -0.823     0.018
RF            -0.156      -0.215    -0.676

--- agg=sum | R2 tranche 10M ---
cond   GDELT/keepall  GDELT/topk  no-GDELT
model                                     
GAT            0.084       0.089     0.117
GCN           -0.027      -0.078     0.050
RF             0.331       0.287    -0.198

--- agg=first | R2 tranche all ---
cond   GDELT/keepall  GDELT/topk  no-GDELT
model                                     
GAT            0.121       0.219     0.108
GCN           -0.101      -0.201    -0.168
RF             0.453       0.416     0.303

--- agg=first | R2 tranche 10M ---
cond   GDELT/keepall  GDELT/topk  no-GDELT
model                                     
GAT           -0.028       0.146    -0.071
GCN           -0.115       0.040    -0.087
RF             0.391       0.326     0.339


In [61]:

#import os
#os.rename("results.csv", "results_old_mixed.csv")   # archive instead of delete